In [1]:
""""# Run the below script or schedule it to run regularly to optimize your Lakehouse table 'Bronze_Raw
from delta.tables import *
deltaTable = DeltaTable.forName(spark, "Bronze_Raw_API")
deltaTable.optimize().executeCompaction()

# If you only want to optimize a subset of your data, you can specify an optional partition predicate. For example:
#
#     from datetime import datetime, timedelta
#     startDate = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')
#     deltaTable.optimize().where("date > '{}'".format(startDate)).executeCompaction()"""


StatementMeta(, 3ac74d2b-d68e-4d93-9099-f688212e8a95, 3, Finished, Available, Finished, False)

AnalysisException: [DELTA_MISSING_DELTA_TABLE] `chimcobldhq2air9e9gms9aiahfl0sjfd9im6t15chh6u`.`Bronze_Raw_API` is not a Delta table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark=SparkSession.builder.getOrCreate()

In [ ]:
silver_stream_df=(spark.readStream.format('delta')
                .option('startingVersion','latest')
                .table('Silver.api_silver_data'))


In [ ]:
gold_stream_df=silver_stream_df.select(
    col('userid').alias('user_id'),
    col('Gender').alias('gender'),
    col('Title').alias('title'),
    col("First").alias('first_name'),
    col('Last').alias('last_name'),
    concat_ws(' ',trim(col('first')),trim(col('last'))).alias('full_name'),
    col('Email').alias('email'),
    col('City').alias('city'),
    col('State').alias('state'),
    col('Country').alias('country'),
    col('Postcode').alias('postcode'),
    col("Latitude").cast('double').alias('latitude'),
    col("Longitude").cast('double').alias('longitude'),
    col('NAT').alias('nationality'),
    col('Age').alias('age'),
    col('Birth_Date').alias('birth_date'),
    col('Registered_Date').alias('registered_date'),
    col('Registered_Age').cast('int').alias('registered_age'),
    col('Phone_Number').alias('phone_number'),
    col('Cell_Number').alias('cell_number'),
    col('Injestion_Timestamp').alias("injection_timestamp"),
    col('Processing_Timestamp').alias('processing_timestamp')

)

In [ ]:
def upsert_gold_customers(batch_df,batch_id):
    latest_batch_df=batch_df.selectExpr(
        "*",
        "row_number() over(partition by user_id order by injection_timestamp desc) as rn").filter('rn==1').drop('rn')
    gold_table=DeltaTable.forName(spark,'gold.gold_customers')
    (gold_table.alias('target').merge(latest_batch_df.alias('source'),'target.user_id=source.user_id')
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    print(f"Batch {batch_id} processed successfully")


In [ ]:
gold_query=(
    gold_stream_df.
    writeStream.
    foreachBatch(upsert_gold_customers).
    option('checkpointLocation','Files/checkpoints/silver_to_gold_customers').start())
print('Silver --> Gold Stream started successfully')

In [ ]:
quality_stream_df=(
    gold_stream_df
    .withColumn(
        "profile_completeness_score",
        (
            when(col('first_name').isNotNull(),1).otherwise(0)
            +
            when(col('last_name').isNotNull(),1).otherwise(0)
            +
            when(col('email').isNotNull(),1).otherwise(0)
            +
            when(col('phone_number').isNotNull(),1).otherwise(0)
            +
            when(col('country').isNotNull(),1).otherwise(0)
            +
            when(col('city').isNotNull(),1).otherwise(0)
            +
            when(col('age').isNotNull(),1).otherwise(0)
        )
    )
)

In [ ]:
quality_stream_df=(
    quality_stream_df.withColumn(
        "profile_quality",
        when(col('profile_completeness_score')==7,"Complete").
        when(col('profile_completeness_score')>=5,"Mostly Complete").
        when(col('profile_completeness_score')>=3,"Partially Complete").otherwise('Poor')
    )
)

In [ ]:
def upsert_quality(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    latest_batch_df=(
        batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    )
    gold=DeltaTable.forName(spark,'gold.gold_data_quality')
    (gold.alias('t').merge(latest_batch_df.alias('s'),'t.user_id=s.user_id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

In [ ]:
quality_query=(
    quality_stream_df.writeStream
    .foreachBatch(upsert_quality)
    .option('checkpointLocation','Files/checkpoints/quality')
    .start()
)

In [ ]:
metrics_stream_df=(
    gold_stream_df
    .withColumn('processing_latency_seconds',
    unix_timestamp('processing_timestamp')
    -
    unix_timestamp('injection_timestamp'))
)

In [ ]:
def upsert_metrics(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    latest_batch_df=(
        batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    )
    table=DeltaTable.forName(
        spark,'gold.gold_pipeline_metrics'
    )
    (table.alias('target')
    .merge(
        latest_batch_df.alias('source'),
        'target.user_id=source.user_id'
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
    )
    print(f"Metrics batch {batch_id} Processed")

In [ ]:
metrics_query=(
    metrics_stream_df.writeStream
    .foreachBatch(upsert_metrics)
    .option(
        'checkpointLocation',
        'Files/checkpoints/metrics'
    )
    .start()
)

In [ ]:
demographics_stream_df=(
            spark.readStream.format('delta').
            table('silver.api_silver_data')
            .select(
                col('userid').alias('user_id'),
                col('Gender').alias('gender'),
                col('Age').alias('age'),
                col('NAT').alias('nationality'),
                col("Injestion_Timestamp").alias('Injection_Timestamp')
            ))

In [ ]:
def update_demographics(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    new_df=(
    batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    .withColumn(
    'age_group',
    when(col('age')<18,'Under 18').
    when(col('age')<=25,'18-25').
    when(col('age')<=35,'26-35').
    when(col('age')<=50,'36-50').
    otherwise('51+'))
    .select('user_id','gender','age_group','nationality'))
    state=DeltaTable.forName(spark,'gold.gold_demographic_state')
    changes=(
            new_df.alias('n').join(state.toDF().alias('s'),'user_id','left')
            .select(
                col('s.gender').alias('old_gender'),
                col('s.age_group').alias('old_age_group'),
                col('s.nationality').alias('old_nationality'),
                col('n.gender').alias('new_gender'),
                col('n.age_group').alias('new_age_group'),
                col('n.nationality').alias('new_nationality')
                )
            )

    old_changes=(changes
            .filter(col('old_gender').isNotNull())
            .select(
                col('old_gender').alias('gender'),
                col('old_age_group').alias('age_group'),
                col('old_nationality').alias('nationality'),
                lit(-1).alias('delta')
            ))

    new_changes=(changes
            .filter(col('new_gender').isNotNull())
            .select(
                col('new_gender').alias('gender'),
                col('new_age_group').alias('age_group'),
                col('new_nationality').alias('nationality'),
                lit(1).alias('delta')
            ))

    deltas=(
        old_changes.
        unionByName(new_changes)
        .groupBy('gender','age_group','nationality')
        .sum('delta')
        .withColumnRenamed('sum(delta)','delta')
    )

    demo=DeltaTable.forName(spark,'gold.gold_demographics')
    (
        demo.alias('t')
        .merge(deltas.alias('s'),
        """t.gender=s.gender AND t.age_group=s.age_group AND t.nationality=s.nationality""")
        .whenMatchedUpdate(set={'customer_count':"t.customer_count+s.delta"})
        .whenNotMatchedInsert(values={'gender':'s.gender','age_group':'s.age_group',
        'nationality':'s.nationality',
        'customer_count':'s.delta'})
        .execute())
    (
        state.alias('t')
        .merge(new_df.alias('s'),
            't.user_id = s.user_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Demographic batch {batch_id} processed")


In [ ]:
demographics_query=(
    demographics_stream_df.writeStream
    .foreachBatch(update_demographics)
    .option('checkpointLocation','Files/checkpoints/demographics')
    .start()
)

In [ ]:
geography_stream_df=(
    spark.readStream
    .format('delta')
    .table('silver.api_silver_data')
    .select(
        col('userid').alias('user_id'),
        col('Country').alias('country'),
        col('State').alias('state'),
        col('City').alias('city'),
        col("Injestion_Timestamp").alias('injection_timestamp')
    )
)

In [ ]:
def update_geography(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    new_df=(batch_df
            .withColumn('rn',row_number().over(window_spec))
            .filter(col('rn')==1)
            .drop('rn')
            .select('user_id','country','state','city'))
    state=DeltaTable.forName(spark,'gold.gold_geography_state')
    changes=(
        new_df.alias('n')
        .join(state.toDF().alias('s'),'user_id','left')
        .select(
            col('s.country').alias("old_country"),
            col('s.state').alias('old_state'),
            col('s.city').alias('old_city'),
            col('n.country').alias("new_country"),
            col('n.state').alias('new_state'),
            col('n.city').alias('new_city')
        )
    )
    old_changes=(
        changes.filter(col('old_country').isNotNull())
        .select(
            col('old_country').alias('country'),
            col('old_state').alias('state'),
            col('old_city').alias('city'),
            lit(-1).alias('delta')
        )
    )
    new_changes=(
        changes.filter(col('new_country').isNotNull())
        .select(
            col('new_country').alias('country'),
            col('new_state').alias('state'),
            col('new_city').alias('city'),
            lit(1).alias('delta')
        )
    )
    deltas=(
        old_changes
        .unionByName(new_changes)
        .groupBy('country','state','city')
        .sum('delta')
        .withColumnRenamed('sum(delta)','delta')
    )
    geo=DeltaTable.forName(
        spark,'gold.gold_geography'
    )
    (geo.alias('t')
    .merge(deltas.alias('s'),
        "t.country=s.country AND t.state=s.state AND t.city=s.city")
        .whenMatchedUpdate(set={"customer_count":'t.customer_count+s.delta'})
        .whenNotMatchedInsert(
            values={"country":"s.country",
                    "state":"s.state",
                    "city":"s.city",
                    "customer_count":"s.delta"}
        ).execute()
    )
    (state.alias('t')
    .merge(new_df.alias('s'),"t.user_id=s.user_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
    print(f"Geography batch {batch_id} processed")

In [ ]:
geography_query=(
    geography_stream_df.writeStream
    .foreachBatch(update_geography)
    .option('checkpointLocation','Files/checkpoints/geography')
    .start()
)

In [1]:
silver_to_gold_code='''
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
spark=SparkSession.builder.getOrCreate()

silver_stream_df=(spark.readStream.format('delta')
                .option('startingVersion','latest')
                .table('Silver.api_silver_data'))

gold_stream_df=silver_stream_df.select(
    col('userid').alias('user_id'),
    col('Gender').alias('gender'),
    col('Title').alias('title'),
    col("First").alias('first_name'),
    col('Last').alias('last_name'),
    concat_ws(' ',trim(col('first')),trim(col('last'))).alias('full_name'),
    col('Email').alias('email'),
    col('City').alias('city'),
    col('State').alias('state'),
    col('Country').alias('country'),
    col('Postcode').alias('postcode'),
    col("Latitude").cast('double').alias('latitude'),
    col("Longitude").cast('double').alias('longitude'),
    col('NAT').alias('nationality'),
    col('Age').alias('age'),
    col('Birth_Date').alias('birth_date'),
    col('Registered_Date').alias('registered_date'),
    col('Registered_Age').cast('int').alias('registered_age'),
    col('Phone_Number').alias('phone_number'),
    col('Cell_Number').alias('cell_number'),
    col('Injestion_Timestamp').alias("injection_timestamp"),
    col('Processing_Timestamp').alias('processing_timestamp')
)
def upsert_gold_customers(batch_df,batch_id):
    latest_batch_df=batch_df.selectExpr(
        "*",
        "row_number() over(partition by user_id order by injection_timestamp desc) as rn").filter('rn==1').drop('rn')
    gold_table=DeltaTable.forName(spark,'gold.gold_customers')
    (gold_table.alias('target').merge(latest_batch_df.alias('source'),'target.user_id=source.user_id')
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
gold_query=(
    gold_stream_df.
    writeStream.
    foreachBatch(upsert_gold_customers).
    option('checkpointLocation','Files/checkpoints/silver_to_gold_customers').start())
print('Silver --> Gold Stream started successfully')
quality_stream_df=(
    gold_stream_df
    .withColumn(
        "profile_completeness_score",
        (
            when(col('first_name').isNotNull(),1).otherwise(0)
            +
            when(col('last_name').isNotNull(),1).otherwise(0)
            +
            when(col('email').isNotNull(),1).otherwise(0)
            +
            when(col('phone_number').isNotNull(),1).otherwise(0)
            +
            when(col('country').isNotNull(),1).otherwise(0)
            +
            when(col('city').isNotNull(),1).otherwise(0)
            +
            when(col('age').isNotNull(),1).otherwise(0)
        )
    )
)
quality_stream_df=(
    quality_stream_df.withColumn(
        "profile_quality",
        when(col('profile_completeness_score')==7,"Complete").
        when(col('profile_completeness_score')>=5,"Mostly Complete").
        when(col('profile_completeness_score')>=3,"Partially Complete").otherwise('Poor')
    )
)
def upsert_quality(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    latest_batch_df=(
        batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    )
    gold=DeltaTable.forName(spark,'gold.gold_data_quality')
    (gold.alias('t').merge(latest_batch_df.alias('s'),'t.user_id=s.user_id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
quality_query=(
    quality_stream_df.writeStream
    .foreachBatch(upsert_quality)
    .option('checkpointLocation','Files/checkpoints/quality')
    .start()
)
metrics_stream_df=(
    gold_stream_df
    .withColumn('processing_latency_seconds',
    unix_timestamp('processing_timestamp')
    -
    unix_timestamp('injection_timestamp'))
)
def upsert_metrics(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    latest_batch_df=(
        batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    )
    table=DeltaTable.forName(
        spark,'gold.gold_pipeline_metrics'
    )
    (table.alias('target')
    .merge(
        latest_batch_df.alias('source'),
        'target.user_id=source.user_id'
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
    )
metrics_query=(
    metrics_stream_df.writeStream
    .foreachBatch(upsert_metrics)
    .option(
        'checkpointLocation',
        'Files/checkpoints/metrics'
    )
    .start()
)
demographics_stream_df=(
            spark.readStream.format('delta').
            table('silver.api_silver_data')
            .select(
                col('userid').alias('user_id'),
                col('Gender').alias('gender'),
                col('Age').alias('age'),
                col('NAT').alias('nationality'),
                col("Injestion_Timestamp").alias('Injection_Timestamp')
            ))
def update_demographics(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    new_df=(
    batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    .withColumn(
    'age_group',
    when(col('age')<18,'Under 18').
    when(col('age')<=25,'18-25').
    when(col('age')<=35,'26-35').
    when(col('age')<=50,'36-50').
    otherwise('51+'))
    .select('user_id','gender','age_group','nationality'))
    state=DeltaTable.forName(spark,'gold.gold_demographic_state')
    changes=(
            new_df.alias('n').join(state.toDF().alias('s'),'user_id','left')
            .select(
                col('s.gender').alias('old_gender'),
                col('s.age_group').alias('old_age_group'),
                col('s.nationality').alias('old_nationality'),
                col('n.gender').alias('new_gender'),
                col('n.age_group').alias('new_age_group'),
                col('n.nationality').alias('new_nationality')
                )
            )

    old_changes=(changes
            .filter(col('old_gender').isNotNull())
            .select(
                col('old_gender').alias('gender'),
                col('old_age_group').alias('age_group'),
                col('old_nationality').alias('nationality'),
                lit(-1).alias('delta')
            ))

    new_changes=(changes
            .filter(col('new_gender').isNotNull())
            .select(
                col('new_gender').alias('gender'),
                col('new_age_group').alias('age_group'),
                col('new_nationality').alias('nationality'),
                lit(1).alias('delta')
            ))

    deltas=(
        old_changes.
        unionByName(new_changes)
        .groupBy('gender','age_group','nationality')
        .sum('delta')
        .withColumnRenamed('sum(delta)','delta')
    )

    demo=DeltaTable.forName(spark,'gold.gold_demographics')
    (
        demo.alias('t')
        .merge(deltas.alias('s'),
        """t.gender=s.gender AND t.age_group=s.age_group AND t.nationality=s.nationality""")
        .whenMatchedUpdate(set={'customer_count':"t.customer_count+s.delta"})
        .whenNotMatchedInsert(values={'gender':'s.gender','age_group':'s.age_group',
        'nationality':'s.nationality',
        'customer_count':'s.delta'})
        .execute())
    (
        state.alias('t')
        .merge(new_df.alias('s'),
            't.user_id = s.user_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Demographic batch {batch_id} processed")
demographics_query=(
    demographics_stream_df.writeStream
    .foreachBatch(update_demographics)
    .option('checkpointLocation','Files/checkpoints/demographics')
    .start()
)
geography_stream_df=(
    spark.readStream
    .format('delta')
    .table('silver.api_silver_data')
    .select(
        col('userid').alias('user_id'),
        col('Country').alias('country'),
        col('State').alias('state'),
        col('City').alias('city'),
        col("Injestion_Timestamp").alias('injection_timestamp')
    )
)
def update_geography(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    new_df=(batch_df
            .withColumn('rn',row_number().over(window_spec))
            .filter(col('rn')==1)
            .drop('rn')
            .select('user_id','country','state','city'))
    state=DeltaTable.forName(spark,'gold.gold_geography_state')
    changes=(
        new_df.alias('n')
        .join(state.toDF().alias('s'),'user_id','left')
        .select(
            col('s.country').alias("old_country"),
            col('s.state').alias('old_state'),
            col('s.city').alias('old_city'),
            col('n.country').alias("new_country"),
            col('n.state').alias('new_state'),
            col('n.city').alias('new_city')
        )
    )
    old_changes=(
        changes.filter(col('old_country').isNotNull())
        .select(
            col('old_country').alias('country'),
            col('old_state').alias('state'),
            col('old_city').alias('city'),
            lit(-1).alias('delta')
        )
    )
    new_changes=(
        changes.filter(col('new_country').isNotNull())
        .select(
            col('new_country').alias('country'),
            col('new_state').alias('state'),
            col('new_city').alias('city'),
            lit(1).alias('delta')
        )
    )
    deltas=(
        old_changes
        .unionByName(new_changes)
        .groupBy('country','state','city')
        .sum('delta')
        .withColumnRenamed('sum(delta)','delta')
    )
    geo=DeltaTable.forName(
        spark,'gold.gold_geography'
    )
    (geo.alias('t')
    .merge(deltas.alias('s'),
        "t.country=s.country AND t.state=s.state AND t.city=s.city")
        .whenMatchedUpdate(set={"customer_count":'t.customer_count+s.delta'})
        .whenNotMatchedInsert(
            values={"country":"s.country",
                    "state":"s.state",
                    "city":"s.city",
                    "customer_count":"s.delta"}
        ).execute()
    )
    (state.alias('t')
    .merge(new_df.alias('s'),"t.user_id=s.user_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
    print(f"Geography batch {batch_id} processed")
geography_query=(
    geography_stream_df.writeStream
    .foreachBatch(update_geography)
    .option('checkpointLocation','Files/checkpoints/geography')
    .start()
)
geography_query.awaitTermination()
'''

notebookutils.fs.put("Files/silver_to_gold.py",silver_to_gold_code,True)
print('Production Python file created succesfully')

StatementMeta(, 1eac1cdc-e1fe-4abd-8098-3b2433241303, 3, Finished, Available, Finished, False)

Production Python file created succesfully


In [1]:
bronze_code=notebookutils.fs.head('Files/bronze_to_silver.py',500000)
silver_code=notebookutils.fs.head('Files/silver_to_gold.py',1000000)
bronze_code=bronze_code.replace("query.awaitTermination()","")
silver_code=silver_code.replace("geography_query.awaitTermination()","")
combined_code=(
                bronze_code
                +"\n\n"
                +silver_code
                +"\n\n"
                +"# keep the combined production streaming application alive"+"spark."
                +"spark.streams.awaitTermination()\n")
notebookutils.fs.put("Files/rt_project_streaming_prd.py",combined_code,True)
print("combined")

StatementMeta(, 2574deb9-679a-444a-8a08-f4ae3d7384ab, 3, Finished, Available, Finished, False)

combined


In [2]:
print(notebookutils.fs.ls("Files/"))

StatementMeta(, 2574deb9-679a-444a-8a08-f4ae3d7384ab, 4, Finished, Available, Finished, False)

[FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/bronze_to_silver.py, name=bronze_to_silver.py, size=3781), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/checkpoint, name=checkpoint, size=0), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/checkpoints, name=checkpoints, size=0), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/rt_project_streaming_prd.py, name=rt_project_streaming_prd.py, size=14317), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/silver_to_gold.py, name=silver_to_gold.py, size=10493)]
